# SHIPP — Validate Agent `save_item` Boundary

**Review draft. WRITE TEST IS DISABLED BY DEFAULT.**

This notebook validates the production boundary:

`Gold candidate → current Lakebase status → idempotent saved_items write → agent_activity audit`

## Safety
The notebook starts in **read-only preflight mode**.

Set:

```python
RUN_WRITE_TEST = True
```

only after every preflight gate passes and you intentionally want to create
one test `saved_items` row plus its audit record.

## Current contract checks
- Gold columns match the Agent's `CANDIDATE_MATCH_COLUMNS`.
- `saved_items` has the unique `(user_id, request_id, listing_id)` constraint.
- `agent_activity.action_status` accepts the exact values emitted by the Agent.
- Required Lakebase connection settings are available.

In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
RUN_WRITE_TEST = False  # SAFETY SWITCH — keep False during review

agent_src = os.path.join(REPO_ROOT, "agent", "agent_ship", "src")
if agent_src not in sys.path:
    sys.path.insert(0, agent_src)

from shipp.agent.contracts import ActionStatus, CANDIDATE_MATCH_COLUMNS
from shipp.agent.lakebase import LakebaseRepo
from shipp.agent.config import AgentSettings
from shipp.agent.clients import build_lakebase_connect, build_workspace_client

print("RUN_WRITE_TEST =", RUN_WRITE_TEST)

In [ ]:
# Gate 1 — Gold must be consumable by the Agent before any write test.
assert table_exists(GOLD_CANDIDATE_MATCHES), f"Missing {GOLD_CANDIDATE_MATCHES}"
gold = spark.table(GOLD_CANDIDATE_MATCHES)

missing_agent_columns = [
    c for c in CANDIDATE_MATCH_COLUMNS if c not in gold.columns
]
print("Missing Agent Gold columns:", missing_agent_columns or "none")
assert not missing_agent_columns, (
    "BLOCKED: fix the Gold ↔ Agent schema contract before testing save_item. "
    f"Missing: {missing_agent_columns}"
)

In [ ]:
# Gate 2 — Lakebase connection environment.
required_pg_env = ["PGHOST", "PGDATABASE", "PGUSER"]
missing_pg_env = [name for name in required_pg_env if not os.getenv(name)]

print("Lakebase environment:")
for name in required_pg_env:
    print(f"  {name}: {'SET' if os.getenv(name) else 'MISSING'}")
print("  SHIPP_LAKEBASE_INSTANCE:", "SET" if os.getenv("SHIPP_LAKEBASE_INSTANCE") else "not set")
print("  PGPASSWORD:", "SET" if os.getenv("PGPASSWORD") else "not set")

assert not missing_pg_env, (
    "BLOCKED: notebook compute does not have the required Lakebase connection environment: "
    + ", ".join(missing_pg_env)
)
assert os.getenv("PGPASSWORD") or os.getenv("SHIPP_LAKEBASE_INSTANCE"), (
    "BLOCKED: provide either PGPASSWORD or SHIPP_LAKEBASE_INSTANCE for OAuth credential generation."
)

In [ ]:
# Build only the connection/repository pieces needed for save_item validation.
settings = AgentSettings(
    llm_endpoint=os.getenv("SHIPP_LLM_ENDPOINT", "unused-for-save-validation"),
    sql_warehouse_id=os.getenv("SHIPP_SQL_WAREHOUSE_ID", "unused-for-save-validation"),
    candidate_matches_table=GOLD_CANDIDATE_MATCHES,
    search_index=os.getenv(
        "SHIPP_SEARCH_INDEX",
        f"{CATALOG}.{GOLD_SCHEMA}.gold_listing_search_docs_index",
    ),
    lakebase_schema=os.getenv("SHIPP_LAKEBASE_SCHEMA", "shipp"),
    lakebase_instance=os.getenv("SHIPP_LAKEBASE_INSTANCE"),
)

workspace = build_workspace_client()
connect = build_lakebase_connect(settings, workspace)
lakebase = LakebaseRepo(connect, settings.lakebase_schema)
print("Lakebase schema:", settings.lakebase_schema)

In [ ]:
# Gate 3 — inspect the real PostgreSQL constraints without changing data.
with connect() as conn, conn.cursor() as cur:
    cur.execute(
        '''
        SELECT conname, pg_get_constraintdef(oid)
        FROM pg_constraint
        WHERE conrelid = %s::regclass
        ORDER BY conname
        ''',
        (f"{settings.lakebase_schema}.saved_items",),
    )
    saved_constraints = dict(cur.fetchall())

    cur.execute(
        '''
        SELECT conname, pg_get_constraintdef(oid)
        FROM pg_constraint
        WHERE conrelid = %s::regclass
        ORDER BY conname
        ''',
        (f"{settings.lakebase_schema}.agent_activity",),
    )
    activity_constraints = dict(cur.fetchall())

print("saved_items constraints:")
for name, definition in saved_constraints.items():
    print(" ", name, "=>", definition)

print("\nagent_activity constraints:")
for name, definition in activity_constraints.items():
    print(" ", name, "=>", definition)

In [ ]:
unique_def = saved_constraints.get("uk_saved_item_user_request_listing", "")
assert "UNIQUE" in unique_def.upper(), (
    "BLOCKED: migration 009 unique saved-item constraint is not present."
)

action_status_def = next(
    (definition for name, definition in activity_constraints.items()
     if "action_status" in definition),
    ""
)

required_action_values = [status.value for status in ActionStatus]
missing_action_values = [
    value for value in required_action_values
    if f"'{value}'" not in action_status_def
]

print("Agent emits action_status values:", required_action_values)
print("Missing exact DB-accepted values:", missing_action_values or "none")

assert not missing_action_values, (
    "BLOCKED: agent_activity.action_status is incompatible with the Agent. "
    f"DB constraint must accept exact values: {required_action_values}"
)

In [ ]:
# Select one deterministic Gold candidate for a future intentional write test.
# No write happens in this cell.
candidate = (
    gold.orderBy("request_id", "match_rank", "listing_id")
    .select("request_id", "listing_id", "match_score")
    .first()
)
assert candidate is not None, "BLOCKED: no Gold candidate exists."

REQUEST_ID = str(candidate["request_id"])
LISTING_ID = str(candidate["listing_id"])

with connect() as conn, conn.cursor() as cur:
    cur.execute(
        f"SELECT requester_id, status FROM {settings.lakebase_schema}.requests WHERE request_id = %s",
        (REQUEST_ID,),
    )
    request_row = cur.fetchone()

    cur.execute(
        f"SELECT status, available_until FROM {settings.lakebase_schema}.listings WHERE listing_id = %s",
        (LISTING_ID,),
    )
    listing_row = cur.fetchone()

assert request_row is not None, "Gold request_id is missing from Lakebase."
assert listing_row is not None, "Gold listing_id is missing from Lakebase."

USER_ID = str(request_row[0])

print("Candidate selected for optional write test:")
print("  user_id:    ", USER_ID)
print("  request_id: ", REQUEST_ID)
print("  listing_id: ", LISTING_ID)
print("  request status:", request_row[1])
print("  listing status:", listing_row[0])

In [ ]:
if not RUN_WRITE_TEST:
    print(
        "REVIEW MODE: no write executed. "
        "After all blockers are fixed, set RUN_WRITE_TEST=True and rerun this cell only intentionally."
    )
else:
    before = None
    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            f'''
            SELECT COUNT(*)
            FROM {settings.lakebase_schema}.saved_items
            WHERE user_id = %s AND request_id = %s AND listing_id = %s
            ''',
            (USER_ID, REQUEST_ID, LISTING_ID),
        )
        before = cur.fetchone()[0]

    first = lakebase.save_item(
        user_id=USER_ID,
        request_id=REQUEST_ID,
        listing_id=LISTING_ID,
    )
    second = lakebase.save_item(
        user_id=USER_ID,
        request_id=REQUEST_ID,
        listing_id=LISTING_ID,
    )

    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            f'''
            SELECT COUNT(*)
            FROM {settings.lakebase_schema}.saved_items
            WHERE user_id = %s AND request_id = %s AND listing_id = %s
            ''',
            (USER_ID, REQUEST_ID, LISTING_ID),
        )
        after = cur.fetchone()[0]

        cur.execute(
            f'''
            SELECT tool_name, entity_id, action_status, created_at
            FROM {settings.lakebase_schema}.agent_activity
            WHERE user_id = %s AND entity_id = %s AND tool_name = 'save_item'
            ORDER BY created_at DESC
            LIMIT 10
            ''',
            (USER_ID, LISTING_ID),
        )
        activity = cur.fetchall()

    print("Before saved rows:", before)
    print("First save result: ", first)
    print("Second save result:", second)
    print("After saved rows: ", after)
    print("Recent audit rows:", activity)

    # Idempotency: the pair can exist at most once.
    assert after <= 1, "FAILED: duplicate saved_items rows exist for the same user/request/listing."
    assert first.status.value in {"success", "duplicate"}, first
    assert second.status.value == "duplicate", second

    print("PASS — save_item write is idempotent and audited.")

In [ ]:
summary = {
    "stage": "agent_save_item_validation",
    "write_test_enabled": RUN_WRITE_TEST,
    "gold_contract_ready": not missing_agent_columns,
    "db_action_status_ready": not missing_action_values,
    "candidate": {
        "user_id": USER_ID,
        "request_id": REQUEST_ID,
        "listing_id": LISTING_ID,
    },
    "status": "READY_FOR_WRITE_TEST" if not RUN_WRITE_TEST else "PASS",
}
print(json.dumps(summary, indent=2, default=str))